<a href="https://colab.research.google.com/github/jakegouveia/HelloWorld/blob/master/Multi_Agent_AI_in_Cybersecurity_%E2%80%93_Learner_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent AI in Cybersecurity & Threat Intelligence

Welcome to the hands-on workshop.

In this lab, you will explore how specialist AI agents can work together
inside a Security Operations Center (SOC).

We will work with:

- Threat Detection Agent
- Threat Intelligence Agent
- Investigation Agent
- Containment Agent
- MCP — Agent ↔ Tools/Data
- A2A concepts — Agent ↔ Agent
- Human-in-the-Loop approval

No API keys or local environment setup are required.

In [ ]:
import requests
import uuid
import json

### Workshop connection

In [ ]:
WORKSHOP_API = "https://udacity-cyber-workshop-api-production.up.railway.app"

WORKSHOP_TOKEN = input(
    "Enter workshop access code: "
)

PARTICIPANT_ID = str(uuid.uuid4())[:8]

print("Participant ID:", PARTICIPANT_ID)

Enter workshop access code: UDACITY-SOC-2026
Participant ID: 4576a7db


### Test the backend from Colab

In [ ]:
response = requests.get(
    f"{WORKSHOP_API}/health",
    timeout=30
)

print(response.json())

{'status': 'ok', 'service': 'Udacity Cybersecurity Workshop API', 'model': 'gpt-5.6-luna', 'workshop_enabled': True}


### Common API helper

In [ ]:
def workshop_post(endpoint, payload):

    response = requests.post(
        f"{WORKSHOP_API}{endpoint}",
        headers={
            "X-Workshop-Token": WORKSHOP_TOKEN,
            "X-Participant-ID": PARTICIPANT_ID
        },
        json=payload,
        timeout=120
    )

    if not response.ok:
        print("API Error:", response.status_code)
        print(response.text)
        response.raise_for_status()

    return response.json()

### Incident

# Exercise 1 — Analyze the Incident

A privileged Finance account has experienced repeated failed logins,
followed by a successful login from an unusual location.

Before using AI, think about:

1. What looks suspicious?
2. Would you classify this LOW, MEDIUM, or HIGH risk?
3. What would you investigate next?

In [ ]:
incident = {
    "user": "finance_admin",
    "department": "Finance",
    "failed_logins": 14,
    "successful_login": True,
    "source_ip": "185.220.101.5",
    "login_country": "Unknown",
    "usual_country": "India",
    "time": "02:13 AM"
}

incident

{'user': 'finance_admin',
 'department': 'Finance',
 'failed_logins': 14,
 'successful_login': True,
 'source_ip': '185.220.101.5',
 'login_country': 'Unknown',
 'usual_country': 'India',
 'time': '02:13 AM'}

### Detection Agent function

# Exercise 2 — Threat Detection Agent

The Detection Agent evaluates whether the incident appears suspicious.

In [ ]:
def run_detection_agent(
    incident,
    extra_instruction=""
):

    response = workshop_post(
        "/detection",
        {
            "incident": incident,
            "extra_instruction": extra_instruction
        }
    )

    return response["result"]

In [ ]:
detection_result = run_detection_agent(
    incident
)

print(detection_result)

1. **Risk Level: HIGH**

2. **Main reasons:**
   - A successful login occurred after **14 failed attempts**, suggesting possible credential guessing or compromise.
   - The **Finance admin** account logged in at **02:13 AM** from an **unknown country**, unlike its usual activity in India.

3. **Recommended next action:**  
   Immediately suspend the account or active session, force a credential reset, and verify the login with the account owner.


## Your Turn

Ask the Detection Agent to also provide a confidence score from 0–100.

In [ ]:
extra_instruction = """
# TODO:
# Ask the agent to provide a confidence score from 0 to 100.
"""

In [ ]:
extra_instruction = """
Also provide a confidence score from 0 to 100.
"""

In [ ]:
detection_result_v2 = run_detection_agent(
    incident,
    extra_instruction
)

print(detection_result_v2)

1. **Risk Level: HIGH**

2. **Two main reasons:**
   - A successful login followed **14 failed login attempts**, suggesting possible credential attacks.
   - The login occurred at **02:13 AM** from an **unknown country**, unlike the user’s usual country, India.

3. **Recommended next action:**  
   Immediately verify the login with the user, review related authentication activity, and consider resetting the account password and enforcing MFA.

**Confidence: 95/100**


# Exercise 3 — What Happens When Context Changes?

Change the security incident and observe how the agent's risk assessment changes.

In [ ]:
modified_incident = incident.copy()

modified_incident["failed_logins"] = 2
modified_incident["login_country"] = "India"
modified_incident["time"] = "11:15 AM"

modified_result = run_detection_agent(
    modified_incident
)

print(modified_result)

1. **Risk Level: MEDIUM**

2. **Two main reasons:**
   - The finance administrator account had **two failed login attempts followed by a successful login**, which may indicate attempted unauthorized access.
   - The account is **privileged and finance-related**, increasing the potential impact if compromised.

3. **Recommended next action:**
   - **Verify the successful login with the user** and review the associated authentication details, including timing and source IP, for legitimacy.


Did the risk assessment change?

Why?

# Exercise 4 — Threat Intelligence Agent

The Detection Agent identified a suspicious source IP.

A specialist Threat Intelligence Agent will now enrich that indicator.

In [ ]:
def run_threat_intelligence_agent(
    indicator,
    extra_instruction=""
):

    return workshop_post(
        "/threat-intelligence",
        {
            "indicator": indicator,
            "extra_instruction": extra_instruction
        }
    )

In [ ]:
threat_response = run_threat_intelligence_agent(
    incident["source_ip"]
)

print(threat_response["result"])

1. **Indicator Reputation:** Suspicious  
2. **Threat Risk:** **HIGH**  
3. **Important Threat Intelligence Findings:**
   - Risk score: 87
   - 23 reported attacks
   - Associated with credential attacks and brute-force activity
   - Tagged as an anonymization network
   - Last seen: 2026-08-29

4. **Likely Threat Activity:** Credential attacks and brute-force attempts conducted through an anonymization network.

5. **Recommended Next Specialist Agent:** Investigation Agent


In [ ]:
print(
    json.dumps(
        threat_response["intelligence"],
        indent=2
    )
)

{
  "reputation": "Suspicious",
  "risk_score": 87,
  "reported_attacks": 23,
  "tags": [
    "credential_attack",
    "brute_force",
    "anonymization_network"
  ],
  "last_seen": "2026-08-29"
}


# Exercise 5 — MCP: Agent to Tools/Data

MCP gives agents a standardized way to access external capabilities.

Think of it as:

**Agent → MCP → Tools / Data**

In [ ]:
mcp_tools = requests.get(
    f"{WORKSHOP_API}/mcp/tools",
    timeout=30
).json()

for tool in mcp_tools["tools"]:
    print(
        "Tool:",
        tool["name"]
    )

    print(
        "Description:",
        tool["description"]
    )

    print("-" * 50)

Tool: search_login_logs
Description: 
Search authentication logs for a specific user.

--------------------------------------------------
Tool: get_user_profile
Description: 
Retrieve account and security context for a user.

--------------------------------------------------
Tool: get_endpoint_activity
Description: 
Retrieve endpoint security activity for a user.

--------------------------------------------------
Tool: lookup_ip_reputation
Description: 
Retrieve simulated threat intelligence for an IP.

--------------------------------------------------


### MCP helper

In [ ]:
def call_mcp_tool(
    tool_name,
    arguments
):

    return workshop_post(
        "/mcp/call",
        {
            "tool_name": tool_name,
            "arguments": arguments
        }
    )

## Call one MCP tool

In [ ]:
mcp_result = call_mcp_tool(
    "search_login_logs",
    {
        "user": "finance_admin"
    }
)

print(
    json.dumps(
        mcp_result["result"],
        indent=2
    )
)

[
  {
    "user": "finance_admin",
    "time": "02:05 AM",
    "source_ip": "185.220.101.5",
    "event": "FAILED_LOGIN"
  },
  {
    "user": "finance_admin",
    "time": "02:07 AM",
    "source_ip": "185.220.101.5",
    "event": "FAILED_LOGIN"
  },
  {
    "user": "finance_admin",
    "time": "02:10 AM",
    "source_ip": "185.220.101.5",
    "event": "FAILED_LOGIN"
  },
  {
    "user": "finance_admin",
    "time": "02:13 AM",
    "source_ip": "185.220.101.5",
    "event": "SUCCESSFUL_LOGIN"
  }
]


## Your Turn

Use MCP to retrieve endpoint activity for `finance_admin`.

Which tool should you use?

In [ ]:
tool_name = "YOUR_CHOICE"

In [ ]:
tool_name = "get_endpoint_activity"

In [ ]:
endpoint_result = call_mcp_tool(
    tool_name,
    {
        "user": "finance_admin"
    }
)

print(
    json.dumps(
        endpoint_result["result"],
        indent=2
    )
)

[
  {
    "time": "02:18 AM",
    "event": "PowerShell execution",
    "severity": "High"
  },
  {
    "time": "02:21 AM",
    "event": "Large file archive created",
    "severity": "Medium"
  }
]


### A2A Concepts

# Exercise 6 — A2A: Agent to Agent

MCP connects an agent to tools and data.

A2A focuses on collaboration between agents.

**MCP**

Agent → Tool

**A2A**

Agent → Agent

In [ ]:
def discover_agent(
    capability
):

    return workshop_post(
        "/a2a/discover",
        {
            "required_capability":
                capability
        }
    )

In [ ]:
agent = discover_agent(
    "ip_reputation"
)

print(
    json.dumps(
        agent,
        indent=2
    )
)

{
  "found": true,
  "required_capability": "ip_reputation",
  "agent_id": "threat_intelligence",
  "agent": {
    "name": "Threat Intelligence Agent",
    "description": "Enriches indicators with threat context.",
    "capabilities": [
      "ioc_enrichment",
      "ip_reputation",
      "threat_context"
    ]
  }
}


## Your Turn

We need an agent that can perform:

`root_cause_analysis`

Which agent will the system discover?

In [ ]:
required_capability = "YOUR_CHOICE"

In [ ]:
required_capability = "root_cause_analysis"

In [ ]:
selected_agent = discover_agent(
    required_capability
)

print(
    selected_agent["agent"]["name"]
)

Investigation Agent


### Demonstrate an A2A-style message

In [ ]:
a2a_message = workshop_post(
    "/a2a/message",
    {
        "from_agent": "threat_intelligence",
        "to_agent": "investigation",
        "task": (
            "Investigate suspected "
            "credential compromise"
        ),
        "payload": {
            "user": incident["user"]
        }
    }
)

print(
    json.dumps(
        a2a_message,
        indent=2
    )
)

{
  "protocol_demo": "A2A-style",
  "from_agent": "threat_intelligence",
  "to_agent": "investigation",
  "task": "Investigate suspected credential compromise",
  "payload": {
    "user": "finance_admin"
  },
  "status": "DELIVERED"
}


### Complete Multi-Agent SOC

# Exercise 7 — Run the Complete Multi-Agent SOC

We will now run the incident through the complete workflow.

Detection Agent  
↓  
Threat Intelligence Agent  
↓  
Investigation Agent  
↓  
Containment Agent  
↓  
Human Approval

In [ ]:
def run_soc_workflow(
    incident,
    detection_instruction="",
    threat_instruction="",
    investigation_instruction="",
    containment_instruction=""
):

    return workshop_post(
        "/workflow",
        {
            "incident":
                incident,

            "detection_instruction":
                detection_instruction,

            "threat_instruction":
                threat_instruction,

            "investigation_instruction":
                investigation_instruction,

            "containment_instruction":
                containment_instruction
        }
    )

In [ ]:
soc_result = run_soc_workflow(
    incident
)

print(
    "Status:",
    soc_result["status"]
)

Status: AWAITING_HUMAN_APPROVAL


In [ ]:
print(
    soc_result["steps"]
    ["1_detection"]
    ["result"]
)

1. **Risk Level: HIGH**

2. **Two main reasons:**
   - A finance administrator account had **14 failed login attempts followed by a successful login**.
   - The login occurred at **02:13 AM from an unknown country**, unlike the user’s usual country of India.

3. **Recommended next action:**
   - **Immediately verify the login with the user and temporarily secure the account** by resetting credentials and revoking active sessions if the login is not recognized. Investigate the associated login activity and source IP.


### Show Threat Intelligence

In [ ]:
print(
    soc_result["steps"]
    ["2_threat_intelligence"]
    ["result"]
)

1. **Indicator Reputation:** Suspicious  
2. **Threat Risk:** **HIGH**  
3. **Important Threat Intelligence Findings:**
   - IP address: `185.220.101.5`
   - Risk score: 87
   - Reported in 23 attacks
   - Associated with credential attacks, brute force, and anonymization network activity
   - Last seen: 2026-08-29
4. **Likely Threat Activity:** Brute-force or credential-based attacks conducted through an anonymization network.  
5. **Recommended Next Specialist Agent:** Investigation Agent


### Show Investigation

In [ ]:
print(
    soc_result["steps"]
    ["3_investigation"]
    ["result"]
)

## 1. Investigation Summary
A privileged Finance account (`finance_admin`) successfully logged in at 02:13 AM from suspicious IP `185.220.101.5`, after reported failed login attempts. The login was outside the user’s normal hours and from an unknown country. Shortly afterward, PowerShell was executed and a large file archive was created, indicating likely post-authentication activity and possible data staging.

## 2. Key Evidence
- 14 failed logins were reported before a successful login; the supplied logs explicitly show 3 failures between 02:05–02:10 AM.
- Successful login at 02:13 AM from `185.220.101.5`.
- User normally logs in from India between 08:00 AM and 08:00 PM.
- Account is privileged and belongs to Finance.
- IP reputation: high risk, score 87; reported in 23 attacks and associated with credential attacks, brute force, and anonymization activity.
- PowerShell executed at 02:18 AM.
- Large file archive created at 02:21 AM.
- No direct evidence confirms data exfiltration.

#

### Show Containment

In [ ]:
print(
    soc_result["steps"]
    ["4_containment"]
    ["result"]
)

1. **Incident Severity**  
   **CRITICAL**

2. **Immediate Recommended Actions**  
   - Obtain approval for high-impact containment actions.  
   - Preserve relevant authentication, PowerShell, archive-creation, and endpoint evidence.  
   - Isolate the suspected endpoint from the network while maintaining forensic access if possible.  
   - Investigate the archive’s contents, location, and any attempted transfers.

3. **Account Actions**  
   - Disable or suspend `finance_admin`.  
   - Revoke active sessions, tokens, and other authentication artifacts.  
   - Reset the account password and require MFA enrollment or re-enrollment if supported.  
   - Review recent activity for unauthorized Finance-system access.

4. **Endpoint Actions**  
   - Quarantine the endpoint associated with the 02:13 AM login.  
   - Stop or contain suspicious PowerShell activity if still active.  
   - Preserve the archive and relevant process, command-line, and file-access evidence.  
   - Scan the endpoint

## Show MCP tools used

In [ ]:
print(
    "Threat Intelligence MCP Tool:"
)

print(
    soc_result["steps"]
    ["2_threat_intelligence"]
    ["mcp_tool_used"]
)

print("\nInvestigation MCP Tools:")

for tool in (
    soc_result["steps"]
    ["3_investigation"]
    ["mcp_tools_used"]
):
    print(" -", tool)

Threat Intelligence MCP Tool:
lookup_ip_reputation

Investigation MCP Tools:
 - search_login_logs
 - get_user_profile
 - get_endpoint_activity


### A2A handoffs

In [ ]:
for handoff in soc_result[
    "a2a_handoffs"
]:

    print(
        handoff["from_agent"],
        "→",
        handoff["to_agent"]
    )

    print(
        "Task:",
        handoff["task"]
    )

    print("-" * 50)

detection → threat_intelligence
Task: Enrich suspicious source IP
--------------------------------------------------
threat_intelligence → investigation
Task: Investigate suspected credential compromise
--------------------------------------------------
investigation → containment
Task: Recommend incident containment actions
--------------------------------------------------


### Human-in-the-Loop

# Exercise 8 — Human-in-the-Loop

The Containment Agent can recommend actions.

It should not automatically execute high-impact security actions.

The SOC analyst retains final control.

In [ ]:
human_decision = "PENDING"

In [ ]:
human_decision = "APPROVE"

In [ ]:
#human_decision = "REJECT"

In [ ]:
if human_decision == "APPROVE":

    print(
        "✓ SOC analyst approved "
        "the containment plan."
    )

    print(
        "In production, approved actions "
        "could now be sent to SOAR, "
        "EDR or IAM systems."
    )


elif human_decision == "REJECT":

    print(
        "✗ SOC analyst rejected "
        "the containment plan."
    )

    print(
        "No security action will "
        "be executed."
    )


else:

    print(
        "⏳ Waiting for SOC analyst approval."
    )

    print(
        "No actions have been executed."
    )

✓ SOC analyst approved the containment plan.
In production, approved actions could now be sent to SOAR, EDR or IAM systems.


# What You Built

You have now worked with a simplified AI-powered Security Operations Center.

## Multi-Agent AI

Different agents specialized in different responsibilities:

Detection Agent  
→ detects suspicious activity

Threat Intelligence Agent  
→ enriches indicators

Investigation Agent  
→ correlates evidence

Containment Agent  
→ recommends response actions


## MCP

MCP connected agents to security capabilities.

**Agent → MCP → Tools / Data**

Examples:

- IP reputation
- Authentication logs
- User profiles
- Endpoint events


## A2A

A2A concepts allowed specialist agents to discover and delegate work.

**Agent → Agent**


## Human-in-the-Loop

AI agents can recommend high-impact actions.

Humans retain control over critical security decisions.